<a href="https://colab.research.google.com/github/SalvadorCM786/PySpark/blob/main/S12_DSACD_completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SparkML: Regresión lineal en PySpark**


# Introducción

Apache Spark es un sistema de computación distribuida de código abierto que se utiliza para el procesamiento y análisis de big data. SparkML es la biblioteca de aprendizaje automático que viene con Spark y que proporciona una gama de algoritmos de clasificación, regresión, agrupamiento, filtrado colaborativo y mucho más.

SparkML se desarrolló para abordar las necesidades de procesamiento de datos a gran escala mediante algoritmos de aprendizaje automático en un entorno distribuido. A medida que los conjuntos de datos han seguido creciendo, las bibliotecas tradicionales de aprendizaje automático como Scikit-learn, excelentes para datos de tamaño pequeño a mediano, podrían no escalar eficazmente. SparkML, con sus capacidades de computación distribuida, permite el procesamiento de big data en un clúster de computadoras, acelerando así significativamente el proceso de aprendizaje automático.

En esencia, SparkML funciona dividiendo los datos en múltiples nodos de un clúster para procesarlos en paralelo. Los resultados se combinan para generar la salida. Este proceso, conocido como MapReduce, permite a SparkML gestionar grandes conjuntos de datos de forma eficiente.


## SparkML vs. Scikit-learn

Si bien SparkML y Scikit-learn son potentes herramientas para el aprendizaje automático, existen algunas diferencias entre ambas:

1. **Escala de datos**: Como se mencionó anteriormente, SparkML está diseñado para computación distribuida a gran escala, lo que lo convierte en una excelente opción para el procesamiento de big data. Scikit-learn, por otro lado, es más adecuado para datos de tamaño pequeño a mediano y no está diseñado para gestionar computación distribuida de forma nativa.

2. **Tipos de datos**: SparkML admite diversos tipos de datos que no están disponibles en Scikit-learn. Por ejemplo, puede trabajar directamente con formatos de datos dispersos, ahorrando una cantidad significativa de memoria y recursos computacionales al trabajar con datos dispersos de alta dimensión.

3. **Algoritmos**: Ambas bibliotecas ofrecen una amplia gama de algoritmos de aprendizaje automático. Sin embargo, Scikit-learn tiene una lista de algoritmos ligeramente más extensa. SparkML está en constante crecimiento y se añaden más algoritmos con cada versión.

4. **Facilidad de uso**: Scikit-learn cuenta con una API sencilla y consistente, lo que la hace muy intuitiva. SparkML, por otro lado, tiene una curva de aprendizaje más pronunciada debido a su naturaleza distribuida y a la necesidad de gestionar particiones y clústeres de datos.

5. **Integración con otras herramientas**: SparkML se integra mejor con herramientas de big data como Hadoop y puede trabajar directamente con datos almacenados en el Sistema de Archivos Distribuidos de Hadoop (HDFS). Scikit-learn no es compatible de forma nativa con la integración con Hadoop.

En conclusión, si bien Scikit-learn sigue siendo una excelente herramienta para las tareas tradicionales de aprendizaje automático, SparkML presenta una clara ventaja en lo que respecta al big data. Al usar SparkML, se puede aprovechar el poder de la computación distribuida para las tareas de aprendizaje automático, lo que lo convierte en una herramienta potente en la era del big data.


# **Dataset**

Monitoreo de Estaciones de Carga para Vehículos Eléctricos y Demanda de Red

Usaremos un dataset masivo que registra las sesiones de carga de vehículos eléctricos y el comportamiento de microrredes solares/eólicas urbanas en distintas zonas de una ciudad inteligente, para crear modelos de predicción: regresión y clasificación.



Estructura del Dataset Único:

* **id_sesion:** Identificador único de la recarga o lectura de nodo.
* **temperatura_ambiente:** Temperatura exterior en °C (afecta la eficiencia de las baterías).
***radiacion_solar_actual:** Radiación solar en W/m² (generación local de paneles).
* **velocidad_viento:** Velocidad del viento en m/s (generación eólica auxiliar).
* **demanda_base_hogares:** Consumo base de la zona residencial/comercial en MW.
* **tipo_zona:** Categórica (Residencial, Industrial, Comercial).

______________________________________________________________________________________      
    

Variables objetivo:

1. Para Regresión:
  
    **energia_entregada_kwh:** Energía total suministrada durante la sesión de carga. Permite predecir cuánta energía requerirá el sistema basándose en el clima y la demanda de la zona.

2. Para Clasificación:

    **nivel_estres_red:** Categórica: Estable, Pico de Demanda, Sobrecarga. Permite clasificar si la red eléctrica está en riesgo de saturación según el comportamiento del entorno.

# Instalación

In [23]:
# Instalar PySpark
!pip install -q findspark pyspark

In [24]:
# Instalar Java (Spark lo necesita)
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

In [25]:
# Descargar Apache Spark
!wget -q https://dlcdn.apache.org/spark/spark-4.2.0/spark-4.2.0-bin-hadoop3.tgz
!tar xf spark-4.2.0-bin-hadoop3.tgz

In [26]:
# Configurar variables de entorno
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-4.2.0-bin-hadoop3"

# **Crear sesión de Spark**

In [27]:
import findspark
findspark.init()

In [28]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Sesion_12').config("spark.sql.ansi.enabled", "false").getOrCreate()
sc = spark.sparkContext

In [29]:
spark.version

'4.2.0'

# **Cargar datos**

In [30]:
# Cargar archivo energía_data.csv
df = spark.read.csv('energia_data.csv', header = True)
df.show(truncate=False)

+------------+--------------------+----------------------+------------------+--------------------+-----------+---------------------+----------------+
|id_sesion   |temperatura_ambiente|radiacion_solar_actual|velocidad_viento  |demanda_base_hogares|tipo_zona  |energia_entregada_kwh|nivel_estres_red|
+------------+--------------------+----------------------+------------------+--------------------+-----------+---------------------+----------------+
|SES_00000000|20.586377567861728  |944.7115659255132     |4.766281826563155 |25.809053490729596  |Comercial  |74.43899190025955    |Estable         |
|SES_00000001|34.7364070846922    |155.29901082564618    |14.149913360326284|14.49870140698358   |Comercial  |35.42465872574626    |Sobrecarga      |
|SES_00000002|23.72836071922837   |663.1588941411532     |14.590157631961146|16.163792583282923  |Industrial |48.87891667078363    |Estable         |
|SES_00000003|15.566860760424575  |946.2468371169745     |3.890377773907033 |53.55694161879211   |Co

In [31]:
len(df.columns)

8

## Valores vacíos

In [32]:
from pyspark.sql.functions import col, sum

# Verificar si existen valores vacíos en las columnas
df.select([sum(col(c).isNull().cast("integer")).alias(c) for c in df.columns]).show()

+---------+--------------------+----------------------+----------------+--------------------+---------+---------------------+----------------+
|id_sesion|temperatura_ambiente|radiacion_solar_actual|velocidad_viento|demanda_base_hogares|tipo_zona|energia_entregada_kwh|nivel_estres_red|
+---------+--------------------+----------------------+----------------+--------------------+---------+---------------------+----------------+
|        0|                   0|                     0|               0|                   0|        0|                    0|               0|
+---------+--------------------+----------------------+----------------+--------------------+---------+---------------------+----------------+



## Duplicados

In [21]:
columnas = df.columns

# Agrupa por todas las columnas y cuenta
conteo_duplicados = df.groupBy(columnas).count()
conteo_duplicados.filter("count>1").count()

0

In [22]:
conteo_duplicados.filter("count>1").show(5)

+---------+--------------------+----------------------+----------------+--------------------+---------+---------------------+----------------+-----+
|id_sesion|temperatura_ambiente|radiacion_solar_actual|velocidad_viento|demanda_base_hogares|tipo_zona|energia_entregada_kwh|nivel_estres_red|count|
+---------+--------------------+----------------------+----------------+--------------------+---------+---------------------+----------------+-----+
+---------+--------------------+----------------------+----------------+--------------------+---------+---------------------+----------------+-----+



In [33]:
df.dtypes

[('id_sesion', 'string'),
 ('temperatura_ambiente', 'string'),
 ('radiacion_solar_actual', 'string'),
 ('velocidad_viento', 'string'),
 ('demanda_base_hogares', 'string'),
 ('tipo_zona', 'string'),
 ('energia_entregada_kwh', 'string'),
 ('nivel_estres_red', 'string')]

##Transformación

In [34]:
from pyspark.sql.functions import col

columnas_a_convertir = ["temperatura_ambiente", "radiacion_solar_actual", "velocidad_viento", "energia_entregada_kwh", "demanda_base_hogares"]

for columnas in columnas_a_convertir:
    df = df.withColumn(columnas, col(columnas).cast("float"))

In [35]:
df.dtypes

[('id_sesion', 'string'),
 ('temperatura_ambiente', 'float'),
 ('radiacion_solar_actual', 'float'),
 ('velocidad_viento', 'float'),
 ('demanda_base_hogares', 'float'),
 ('tipo_zona', 'string'),
 ('energia_entregada_kwh', 'float'),
 ('nivel_estres_red', 'string')]

In [36]:
df.columns

['id_sesion',
 'temperatura_ambiente',
 'radiacion_solar_actual',
 'velocidad_viento',
 'demanda_base_hogares',
 'tipo_zona',
 'energia_entregada_kwh',
 'nivel_estres_red']

In [37]:
df.select("tipo_zona").distinct().show()

+-----------+
|  tipo_zona|
+-----------+
| Industrial|
|  Comercial|
|Residencial|
+-----------+



In [38]:
from pyspark.sql.functions import when

df = df.withColumn("tipo_zona_num",
    when(col("tipo_zona") == "Residencial", 1)
    .when(col("tipo_zona") == "Comercial", 2)
    .when(col("tipo_zona") == "Industrial", 3)
    .otherwise(0))

In [39]:
df.select("nivel_estres_red").distinct().show()

+----------------+
|nivel_estres_red|
+----------------+
|         Estable|
|      Sobrecarga|
| Pico de Demanda|
+----------------+



In [40]:
df = df.withColumn("nivel_estres_num",
    when(col("nivel_estres_red") == "Estable", 0)
    .when(col("nivel_estres_red") == "Pico de Demanda", 1)
    .when(col("nivel_estres_red") == "Sobrecarga", 2)
    .otherwise(None))
df.show()

+------------+--------------------+----------------------+----------------+--------------------+-----------+---------------------+----------------+-------------+----------------+
|   id_sesion|temperatura_ambiente|radiacion_solar_actual|velocidad_viento|demanda_base_hogares|  tipo_zona|energia_entregada_kwh|nivel_estres_red|tipo_zona_num|nivel_estres_num|
+------------+--------------------+----------------------+----------------+--------------------+-----------+---------------------+----------------+-------------+----------------+
|SES_00000000|           20.586378|             944.71155|       4.7662816|           25.809053|  Comercial|            74.438995|         Estable|            2|               0|
|SES_00000001|            34.73641|             155.29901|       14.149914|           14.498701|  Comercial|             35.42466|      Sobrecarga|            2|               2|
|SES_00000002|           23.728361|              663.1589|      14.5901575|           16.163792| Industri

In [41]:
df = df.drop("nivel_estres_red", "tipo_zona", "id_sesion")
df.show()

+--------------------+----------------------+----------------+--------------------+---------------------+-------------+----------------+
|temperatura_ambiente|radiacion_solar_actual|velocidad_viento|demanda_base_hogares|energia_entregada_kwh|tipo_zona_num|nivel_estres_num|
+--------------------+----------------------+----------------+--------------------+---------------------+-------------+----------------+
|           20.586378|             944.71155|       4.7662816|           25.809053|            74.438995|            2|               0|
|            34.73641|             155.29901|       14.149914|           14.498701|             35.42466|            2|               2|
|           23.728361|              663.1589|      14.5901575|           16.163792|            48.878918|            3|               0|
|           15.566861|              946.2468|       3.8903778|           53.556942|            128.74211|            2|               2|
|           26.776823|             89.336

# Análisis exploratorio de los datos



In [42]:
df.summary().show()

+-------+--------------------+----------------------+------------------+--------------------+---------------------+------------------+------------------+
|summary|temperatura_ambiente|radiacion_solar_actual|  velocidad_viento|demanda_base_hogares|energia_entregada_kwh|     tipo_zona_num|  nivel_estres_num|
+-------+--------------------+----------------------+------------------+--------------------+---------------------+------------------+------------------+
|  count|              750000|                750000|            750000|              750000|               750000|            750000|            750000|
|   mean|  22.497554523726144|     499.7337682188772| 8.493823633224647|   35.00565063675054|    81.76022546031443|1.9992093333333334|1.1521746666666666|
| stddev|  10.106965687510913|    259.93142310238005|4.3309748511555055|  14.434801086295904|   27.148210295794556|0.8163914074589398|0.8635183061425897|
|    min|           5.0000196|             50.000065|         1.0000352|    

##Correlación


In [43]:
# Correlación de la variable objetivo con cada atributo
# OJO: usamos 'c' como variable del bucle para NO sobrescribir la función col() de PySpark
target = "energia_entregada_kwh"

for c in df.columns:
    if c == target:
        continue
    corr_value = df.stat.corr(target, c)
    print(f"Correlación con {c}: {corr_value:.4f}")


Correlación con temperatura_ambiente: -0.1883
Correlación con radiacion_solar_actual: 0.1914
Correlación con velocidad_viento: 0.0004
Correlación con demanda_base_hogares: 0.9574
Correlación con tipo_zona_num: -0.0008
Correlación con nivel_estres_num: 0.5930


# **Regresión lineal**


## Separar variable objetivo

In [44]:
df.show()

+--------------------+----------------------+----------------+--------------------+---------------------+-------------+----------------+
|temperatura_ambiente|radiacion_solar_actual|velocidad_viento|demanda_base_hogares|energia_entregada_kwh|tipo_zona_num|nivel_estres_num|
+--------------------+----------------------+----------------+--------------------+---------------------+-------------+----------------+
|           20.586378|             944.71155|       4.7662816|           25.809053|            74.438995|            2|               0|
|            34.73641|             155.29901|       14.149914|           14.498701|             35.42466|            2|               2|
|           23.728361|              663.1589|      14.5901575|           16.163792|            48.878918|            3|               0|
|           15.566861|              946.2468|       3.8903778|           53.556942|            128.74211|            2|               2|
|           26.776823|             89.336

Los algoritmos de Machine Learning en Spark están diseñados bajo una arquitectura distribuida y solo aceptan **una columna de entrada** para las características independientes.

Esa única columna debe contener todas las variables explicativas empaquetadas en forma de un vector numérico.

In [45]:
# VectorAssembler: agrupa múltiples columnas numéricas en una sola columna de tipo vector
from pyspark.ml.feature import VectorAssembler

exclusiones = ['energia_entregada_kwh', 'nivel_estres_num']
atributos = [c for c in df.columns if c not in exclusiones]

# IMPORTANTE: este es el orden en que quedarán los coeficientes del modelo
print('Orden de los atributos:', atributos)

df_rl = (VectorAssembler(inputCols=atributos, outputCol="Atributos")
    .transform(df).select('Atributos', 'energia_entregada_kwh'))
df_rl.show()


Orden de los atributos: ['temperatura_ambiente', 'radiacion_solar_actual', 'velocidad_viento', 'demanda_base_hogares', 'tipo_zona_num']
+--------------------+---------------------+
|           Atributos|energia_entregada_kwh|
+--------------------+---------------------+
|[20.5863780975341...|            74.438995|
|[34.7364082336425...|             35.42466|
|[23.7283611297607...|            48.878918|
|[15.5668611526489...|            128.74211|
|[26.7768230438232...|            112.63597|
|[15.0499496459960...|             30.18515|
|[23.6683406829834...|            89.655334|
|[7.76598834991455...|              94.6573|
|[34.1421279907226...|            82.723526|
|[21.6191902160644...|            105.13801|
|[17.0409622192382...|            49.557613|
|[6.21233797073364...|             97.51943|
|[11.7556495666503...|            56.026047|
|[23.061279296875,...|            90.567955|
|[10.2529783248901...|             84.82813|
|[22.9045810699462...|             108.1713|
|[29.5045

## Entrenamiento y prueba


In [46]:
# Train-test split
train_data, test_data = df_rl.randomSplit([0.7, 0.3], seed=0)

In [47]:
from pyspark.ml.regression import LinearRegression

# Modelo de regresión lineal
lr = LinearRegression(featuresCol='Atributos', labelCol='energia_entregada_kwh')

# Ajuste del modelo al conjunto de entrenamiento
lr_model = lr.fit(train_data)

In [48]:
from pyspark.ml.evaluation import RegressionEvaluator

# Aplicación al conjunto de prueba
predictions = lr_model.transform(test_data)

# Evaluación del modelo: RMSE
evaluator_rmse = RegressionEvaluator(labelCol="energia_entregada_kwh", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)
print(f"Error cuadrático medio (RMSE) = {rmse}")

Error cuadrático medio (RMSE) = 2.9912072060206976


In [49]:
# Evaluación del modelo: R2

evaluator_r2 = RegressionEvaluator(labelCol="energia_entregada_kwh", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)
print(f"Coeficiente de determinación (R2) = {r2}")

Coeficiente de determinación (R2) = 0.98786735005058


In [50]:
# Coeficientes del modelo, emparejados con el nombre de cada atributo
coef = lr_model.coefficients

for nombre, valor in zip(atributos, coef):
    print(f"{nombre}: {valor:.6f}")


temperatura_ambiente: -0.499652
radiacion_solar_actual: 0.019998
velocidad_viento: -0.000297
demanda_base_hogares: 1.799705
tipo_zona_num: 0.002617


In [51]:
# Imprimir intersección
inter = lr_model.intercept
print(f"Intersección: {inter}")

Intersección: 20.00383189720976


In [52]:
df.columns

['temperatura_ambiente',
 'radiacion_solar_actual',
 'velocidad_viento',
 'demanda_base_hogares',
 'energia_entregada_kwh',
 'tipo_zona_num',
 'nivel_estres_num']

* **Demanda base de los hogares:** A mayor consumo base en la zona, mayor será la energía total requerida o suministrada durante la sesión de carga.

* **Temperatura ambiente:** Las temperaturas extremas afectan la eficiencia de las baterías y los sistemas de conducción, o bien implican mayor uso de sistemas de climatización que alteran el flujo de la red.


##Predicción

In [53]:
atributos


['temperatura_ambiente',
 'radiacion_solar_actual',
 'velocidad_viento',
 'demanda_base_hogares',
 'tipo_zona_num']

In [54]:
# Valores de ejemplo para una nueva sesión de carga.
# Cambia estos números por el escenario que quieras predecir.
valores = {
    'temperatura_ambiente':   25.0,   # °C
    'radiacion_solar_actual': 600.0,  # W/m²
    'velocidad_viento':       4.5,    # m/s
    'demanda_base_hogares':   12.0,   # MW
    'tipo_zona_num':          1       # 1=Residencial, 2=Comercial, 3=Industrial
}
valores


{'temperatura_ambiente': 25.0,
 'radiacion_solar_actual': 600.0,
 'velocidad_viento': 4.5,
 'demanda_base_hogares': 12.0,
 'tipo_zona_num': 1}

In [55]:
# Predicción manual: suma de (valor * coeficiente) + intersección
# Se recorre 'atributos' para garantizar que cada valor se multiplique por SU coeficiente
pred = inter
for nombre, c in zip(atributos, coef):
    pred += valores[nombre] * c

print(f"Energía entregada estimada: {pred:.2f} kWh")


Energía entregada estimada: 41.11 kWh


Comprobación: la misma predicción hecha con el modelo de Spark en lugar de a mano. Ambos resultados deben coincidir.

In [56]:
# Construimos un DataFrame de una sola fila con el escenario anterior
fila = [tuple(float(valores[a]) for a in atributos)]
df_nuevo = spark.createDataFrame(fila, atributos)

df_nuevo_vec = VectorAssembler(inputCols=atributos, outputCol='Atributos').transform(df_nuevo)
lr_model.transform(df_nuevo_vec).select('prediction').show()


+-----------------+
|       prediction|
+-----------------+
|41.10892914323172|
+-----------------+



# **Clasificación: nivel de estrés de la red**

Ahora predecimos la variable categórica `nivel_estres_num`:

* `0` = Estable
* `1` = Pico de Demanda
* `2` = Sobrecarga

Al ser tres clases se trata de una **clasificación multiclase**.

## Preparación de los datos

In [57]:
# Revisamos el balance de las clases
df.groupBy('nivel_estres_num').count().orderBy('nivel_estres_num').show()


+----------------+------+
|nivel_estres_num| count|
+----------------+------+
|               0|231242|
|               1|173385|
|               2|345373|
+----------------+------+



In [58]:
# Eliminamos filas sin etiqueta (el otherwise(None) de la transformación)
df_clf_base = df.na.drop(subset=['nivel_estres_num'])
print('Filas disponibles para clasificar:', df_clf_base.count())


Filas disponibles para clasificar: 750000


Los atributos son los mismos que en la regresión. La energía entregada sí se puede usar aquí como variable explicativa, porque ya no es la variable objetivo.

In [59]:
atributos_clf = [c for c in df_clf_base.columns if c != 'nivel_estres_num']
print('Atributos de entrada:', atributos_clf)

df_clf = (VectorAssembler(inputCols=atributos_clf, outputCol='Atributos')
    .transform(df_clf_base).select('Atributos', 'nivel_estres_num'))
df_clf.show(5)


Atributos de entrada: ['temperatura_ambiente', 'radiacion_solar_actual', 'velocidad_viento', 'demanda_base_hogares', 'energia_entregada_kwh', 'tipo_zona_num']
+--------------------+----------------+
|           Atributos|nivel_estres_num|
+--------------------+----------------+
|[20.5863780975341...|               0|
|[34.7364082336425...|               2|
|[23.7283611297607...|               0|
|[15.5668611526489...|               2|
|[26.7768230438232...|               2|
+--------------------+----------------+
only showing top 5 rows


## Entrenamiento y prueba

In [60]:
train_clf, test_clf = df_clf.randomSplit([0.7, 0.3], seed=0)

print('Entrenamiento:', train_clf.count())
print('Prueba:', test_clf.count())


Entrenamiento: 524353
Prueba: 225647


### Regresión logística

In [61]:
from pyspark.ml.classification import LogisticRegression

log_reg = LogisticRegression(featuresCol='Atributos', labelCol='nivel_estres_num')
log_model = log_reg.fit(train_clf)

pred_log = log_model.transform(test_clf)
pred_log.select('nivel_estres_num', 'prediction', 'probability').show(10, truncate=False)


+----------------+----------+--------------------------------------------------------------+
|nivel_estres_num|prediction|probability                                                   |
+----------------+----------+--------------------------------------------------------------+
|0               |0.0       |[0.9162076722611577,0.08150222628294844,0.002290101455893693] |
|2               |1.0       |[0.030883247808041962,0.7575750667397536,0.21154168545220442] |
|0               |0.0       |[0.9934467925373718,0.006491905513746045,6.13019488820948E-5] |
|2               |1.0       |[0.04463561726258649,0.7686783167809333,0.18668606595648007]  |
|2               |1.0       |[0.013868313322531566,0.7177095356800867,0.26842215099738176] |
|2               |1.0       |[0.05035955003230555,0.7734561560613932,0.17618429390630136]  |
|0               |0.0       |[0.9774665162042414,0.022188939929391274,3.445438663672033E-4]|
|0               |0.0       |[0.9358975630037241,0.06254491164259059,0

## Evaluación

* **Accuracy**: porcentaje de predicciones correctas.
* **F1**: equilibra precisión y exhaustividad; es más confiable si las clases están desbalanceadas.

In [62]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

eval_acc = MulticlassClassificationEvaluator(
    labelCol='nivel_estres_num', predictionCol='prediction', metricName='accuracy')
eval_f1 = MulticlassClassificationEvaluator(
    labelCol='nivel_estres_num', predictionCol='prediction', metricName='f1')

print(f"Accuracy = {eval_acc.evaluate(pred_log):.4f}")
print(f"F1       = {eval_f1.evaluate(pred_log):.4f}")


Accuracy = 0.7642
F1       = 0.7562


### Matriz de confusión

Las filas son la clase real y las columnas la clase predicha. La diagonal son los aciertos.

In [63]:
pred_log.groupBy('nivel_estres_num').pivot('prediction').count().orderBy('nivel_estres_num').show()


+----------------+-----+-----+-----+
|nivel_estres_num|  0.0|  1.0|  2.0|
+----------------+-----+-----+-----+
|               0|61298| 4301| 3756|
|               1| 4595|25112|22639|
|               2| 8705| 9212|86029|
+----------------+-----+-----+-----+



### Comparación con Random Forest

La regresión logística solo traza fronteras lineales. Un Random Forest captura relaciones no lineales y suele mejorar el resultado en este tipo de problema.

In [64]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol='Atributos', labelCol='nivel_estres_num', numTrees=50, seed=0)
rf_model = rf.fit(train_clf)

pred_rf = rf_model.transform(test_clf)

print(f"Accuracy = {eval_acc.evaluate(pred_rf):.4f}")
print(f"F1       = {eval_f1.evaluate(pred_rf):.4f}")


Accuracy = 0.9846
F1       = 0.9848


### Importancia de los atributos

Random Forest indica qué tanto aportó cada variable a la clasificación.

In [65]:
importancias = rf_model.featureImportances

for nombre, imp in sorted(zip(atributos_clf, importancias), key=lambda x: -x[1]):
    print(f"{nombre}: {imp:.4f}")


demanda_base_hogares: 0.4826
temperatura_ambiente: 0.3258
energia_entregada_kwh: 0.1898
radiacion_solar_actual: 0.0019
velocidad_viento: 0.0000
tipo_zona_num: 0.0000


## Predicción de un caso nuevo

In [66]:
valores_clf = {
    'temperatura_ambiente':   38.0,
    'radiacion_solar_actual': 150.0,
    'velocidad_viento':       1.0,
    'demanda_base_hogares':   22.0,
    'energia_entregada_kwh':  85.0,
    'tipo_zona_num':          3
}

fila_clf = [tuple(float(valores_clf[a]) for a in atributos_clf)]
df_caso = spark.createDataFrame(fila_clf, atributos_clf)
df_caso = VectorAssembler(inputCols=atributos_clf, outputCol='Atributos').transform(df_caso)

etiquetas = {0: 'Estable', 1: 'Pico de Demanda', 2: 'Sobrecarga'}
resultado = rf_model.transform(df_caso).select('prediction').collect()[0][0]

print(f"Nivel de estrés predicho: {etiquetas[int(resultado)]}")


Nivel de estrés predicho: Sobrecarga
